In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy
!pip install pandas numpy>=2.0 pyarrow rapidfuzz openai tqdm -q
# scispaCy is optional; versions must match or the wheel install fails:
!pip install scispacy==0.5.5 -q
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz -q

In [ ]:
#!/usr/bin/env python3
"""
extract_methodology_and_triples.py  —  Google Colab pipeline
============================================================
Two extractions the downstream papers need, each done THREE ways so you can
cross-validate (regex/dictionary -> scispaCy NLP -> dual-LLM consensus):

  A) METHODOLOGY STAGE  (the "evidence escalator" that powers the Paper-3 JSD
     "trapped upstream" test): detection -> prevalence/survey -> experimental/
     vector-competence -> intervention/control -> modeling -> review/synthesis.

  B) SPECIES x PATHOGEN x HOST TRIPLES  (the Paper-2 gap tensor axes):
     tick_species  x  pathogen  x  host, with a relation label.

WHY THREE LAYERS
  - Dictionary/regex: free, deterministic, runs on all rows; also mines the
    already-semi-structured oa_keywords / oa_mesh / oa_concepts columns.
  - scispaCy: catches entities the dictionary misses; gives sentence spans for
    co-occurrence triples. Optional (install is version-sensitive) — wrapped so
    the pipeline still runs without it.
  - LLM (OpenAI + DeepSeek): best at the RELATION (who transmits/harbors what in
    which host) and at stage nuance. Expensive -> run on the genuine-tick subset
    or a sample; fully resumable (cached per oa_id) so you never pay twice.

Reconciliation: an item is "high-confidence" when >=2 layers agree.

COLAB SETUP
  !pip install pandas numpy pyarrow rapidfuzz openai tqdm
  # scispaCy (optional; match versions or it will fail):
  !pip install scispacy==0.5.5
  !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz
  # secrets: add OPENAI_API_KEY and DEEPSEEK_API_KEY in the Colab key panel
"""

import json, os, re, sys, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd

# =========================================================================== #
# 0. CONFIG                                                                    #
# =========================================================================== #
IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_study_location.parquet"
OUT_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_methods_triples.parquet"
LLM_CACHE    = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_extract_cache.jsonl"  # resumable
RUN_LLM      = True          # set False to run only dictionary + scispaCy
LLM_ON       = "tick_only"   # "tick_only" (llm_label==TICK_RELEVANT) | "all" | "sample"
LLM_SAMPLE_N = 1500
LLM_WORKERS  = 8


# =========================================================================== #
# 1. GAZETTEERS + LEXICONS  (extend freely — coverage drives recall)          #
# =========================================================================== #
TICK_GENERA = {
    "ixodes","amblyomma","rhipicephalus","hyalomma","dermacentor","haemaphysalis",
    "boophilus","argas","ornithodoros","otobius","nuttalliella","bothriocroton",
    "anomalohimalaya","cosmiomma","margaropus","nosomma","rhipicentor","antricola",
    "carios","dermacentroma","ixodida",
}
PATHOGEN_TERMS = {  # genus or disease-level; binomials caught by regex too
    "borrelia","rickettsia","anaplasma","ehrlichia","babesia","theileria",
    "coxiella","francisella","bartonella","hepatozoon","cytauxzoon","midichloria",
    "borreliella","neoehrlichia","candidatus",
    "lyme","borreliosis","spotted fever","tick-borne encephalitis","tbev",
    "crimean-congo","cchf","cchfv","heartwater","cowdriosis","anaplasmosis",
    "ehrlichiosis","babesiosis","theileriosis","east coast fever","tularemia",
    "q fever","louping ill","powassan","sftsv","severe fever with thrombocytopenia",
    "nairobi sheep disease","african swine fever","kyasanur","rocky mountain spotted fever",
}
HOST_TERMS = {
    "human":"human","humans":"human","patient":"human","patients":"human","people":"human",
    "cattle":"cattle","bovine":"cattle","cow":"cattle","cows":"cattle","calf":"cattle","calves":"cattle",
    "dog":"dog","dogs":"dog","canine":"dog","canis":"dog",
    "cat":"cat","cats":"cat","feline":"cat",
    "sheep":"sheep","ovine":"sheep","lamb":"sheep",
    "goat":"goat","goats":"goat","caprine":"goat",
    "horse":"horse","horses":"horse","equine":"horse",
    "camel":"camel","camels":"camel","dromedary":"camel",
    "deer":"deer","roe deer":"deer","red deer":"deer","white-tailed deer":"deer","cervid":"deer",
    "rodent":"rodent","rodents":"rodent","mouse":"rodent","mice":"rodent","rat":"rodent","rats":"rodent","vole":"rodent",
    "bird":"bird","birds":"bird","avian":"bird","poultry":"bird","chicken":"bird",
    "wildlife":"wildlife","wild boar":"wildlife","boar":"wildlife","hedgehog":"wildlife",
    "hare":"wildlife","rabbit":"wildlife","buffalo":"buffalo","antelope":"wildlife",
    "reptile":"reptile","lizard":"reptile","tortoise":"reptile","snake":"reptile",
    "livestock":"livestock","small ruminant":"livestock","ruminant":"livestock",
}

# methodology stage: (label, ordered priority, cue regex). Higher priority wins as
# the PRIMARY stage; all matched labels are also kept as a set.
STAGE_CUES = [
    ("review",       6, r"\b(systematic review|meta-?analysis|scoping review|literature review|we review|this review)\b"),
    ("modeling",     5, r"\b(model(l)?ing|predict(ion|ive)?|niche model|species distribution model|maxent|risk map|climate projection|forecast|spatial analysis|machine learning)\b"),
    ("intervention", 4, r"\b(acaricide|ixodicide|vaccine|vaccination|control (strategy|program|measure)|efficacy|treatment|randomi[sz]ed|clinical trial|repellent|resistance)\b"),
    ("experimental", 3, r"\b(vector competence|experimental (infection|transmission)|transmission experiment|acquisition|artificial feeding|life cycle|laboratory (colony|rearing)|co-?feeding)\b"),
    ("prevalence",   2, r"\b(prevalence|sero-?prevalence|cross-?sectional|survey|infestation|distribution|occurrence|epidemiolog|abundance)\b"),
    ("detection",    1, r"\b(detect(ed|ion)?|identif(y|ied|ication)|first (report|record)|molecular characteri|pcr|sequenc(e|ing)|16s|phylogen|morpholog|new species|redescription)\b"),
]

_BINOMIAL = re.compile(r"\b([A-Z][a-z]{3,})\s+([a-z]{3,})\b")  # Genus species


# =========================================================================== #
# 2. LAYER 1 — dictionary / regex (whole corpus, free)                        #
# =========================================================================== #
def _txt(row):
    parts = []
    for c in ("oa_title","oa_abstract","oa_keywords"):
        val = row.get(c)
        parts.append(str(val) if pd.notna(val) else "")
    return " ".join(parts)

def _semi_structured(row):
    """entities already listed in curated columns — very high precision."""
    extracted_parts = []
    for c in ("oa_keywords","oa_mesh","oa_concepts"):
        val = row.get(c)
        if pd.notna(val):
            extracted_parts.append(str(val))
    return " | ".join(extracted_parts).lower()

def dict_entities(row):
    text = _txt(row); low = text.lower(); semi = _semi_structured(row)
    hay = low + " || " + semi
    ticks, paths, hosts = set(), set(), set()

    for g in TICK_GENERA:
        if re.search(rf"\b{re.escape(g)}\b", hay): ticks.add(g)
    for m in _BINOMIAL.finditer(text):            # Genus species binomials
        if m.group(1).lower() in TICK_GENERA:
            ticks.add(f"{m.group(1).lower()} {m.group(2).lower()}")
    for p in PATHOGEN_TERMS:
        if re.search(rf"\b{re.escape(p)}\b", hay): paths.add(p)
    for h, canon in HOST_TERMS.items():
        if re.search(rf"\b{re.escape(h)}\b", hay): hosts.add(canon)

    return ticks, paths, hosts

def dict_methodology(row):
    low = _txt(row).lower()
    if str(row.get("oa_type","")).lower() in ("review",):     # OpenAlex type
        matched = {"review"}
    else:
        matched = set()
    best = (None, -1)
    for label, prio, rx in STAGE_CUES:
        if re.search(rx, low):
            matched.add(label)
            if prio > best[1]:
                best = (label, prio)
    primary = "review" if "review" in matched and best[0] is None else best[0]
    return primary or "other", matched or {"other"}

def dict_triples(row):
    """sentence-level co-occurrence -> candidate triples with a relation cue."""
    abstract_val = row.get("oa_abstract")
    ab = str(abstract_val) if pd.notna(abstract_val) else ""
    triples = []
    REL = [("transmits", r"transmit|vector of|competent vector"),
           ("harbors",  r"harbou?r|carr(y|ied)|positive for|infected with|reservoir"),
           ("detected", r"detect|identif|isolat|present in|found in"),
           ("infests",  r"infest|parasiti[sz]|feed(ing)? on|collected from|removed from")]
    for sent in re.split(r"(?<=[.!?])\s+", ab):
        s = sent.lower()
        t = {g for g in TICK_GENERA if re.search(rf"\b{re.escape(g)}\b", s)}
        p = {x for x in PATHOGEN_TERMS if re.search(rf"\b{re.escape(x)}\b", s)}
        h = {HOST_TERMS[x] for x in HOST_TERMS if re.search(rf"\b{re.escape(x)}\b", s)}
        rel = next((name for name, rx in REL if re.search(rx, s)), "cooccur")
        # emit the cross-product present in THIS sentence (partial triples allowed)
        for ti in (t or {""}):
            for pi in (p or {""}):
                for hi in (h or {""}):
                    if ti or pi or hi:
                        triples.append({"tick":ti,"pathogen":pi,"host":hi,
                                        "relation":rel,"src":"dict"})
    return triples


# =========================================================================== #
# 3. LAYER 2 — scispaCy (optional)                                            #
# =========================================================================== #
_NLP = None
def get_nlp():
    global _NLP
    if _NLP is not None:
        return _NLP
    try:
        import spacy
        _NLP = spacy.load("en_ner_bionlp13cg_md")   # has ORGANISM entity type
        print("[scispacy] loaded en_ner_bionlp13cg_md", file=sys.stderr)
    except Exception as e:
        print(f"[scispacy] unavailable ({e}); skipping NLP layer", file=sys.stderr)
        _NLP = False
    return _NLP

def nlp_entities(text):
    nlp = get_nlp()
    if not nlp or not isinstance(text, str) or not text:
        return set()
    orgs = set()
    for ent in nlp(text[:100000]).ents:
        if ent.label_ in ("ORGANISM", "TAXON", "CELL", "CANCER") or ent.label_ == "ORGANISM":
            orgs.add(ent.text.lower())
    return orgs  # organisms → reconcile against tick/host gazetteers downstream


# =========================================================================== #
# 4. LAYER 3 — dual LLM (OpenAI + DeepSeek), resumable                        #
# =========================================================================== #
SCHEMA_HINT = (
    'Return ONLY compact JSON, no prose, no markdown fences:\n'
    '{"methodology_stage":"detection|prevalence|experimental|intervention|modeling|review|other",'
    '"triples":[{"tick_species":"","pathogen":"","host":"","relation":"transmits|harbors|detected|infests|none"}]}\n'
    'Use "" for an axis not stated. tick_species lowercase binomial or genus. '
    'pathogen lowercase genus/species or disease. host a simple common noun '
    '(human, cattle, dog, sheep, rodent, deer, bird, wildlife, ...). Do not invent.'
)

def _client(provider):
    from openai import OpenAI
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY" if provider=="openai" else "DEEPSEEK_API_KEY")
    except Exception:
        key = os.environ.get("OPENAI_API_KEY" if provider=="openai" else "DEEPSEEK_API_KEY")
    if provider == "deepseek":
        return OpenAI(api_key=key, base_url="https://api.deepseek.com"), "deepseek-chat"
    return OpenAI(api_key=key), "gpt-4o-mini"

_CLIENTS = {}
def call_llm(provider, title, abstract):
    if provider not in _CLIENTS:
        _CLIENTS[provider] = _client(provider)
    client, model = _CLIENTS[provider]
    user = f"TITLE: {title}\n\nABSTRACT: {abstract}\n\n{SCHEMA_HINT}"
    for attempt in range(4):
        try:
            r = client.chat.completions.create(
                model=model, temperature=0,
                messages=[{"role":"system","content":"You extract structured data from tick/tick-borne-disease abstracts."},
                          {"role":"user","content":user}],
                max_tokens=700)
            txt = r.choices[0].message.content.strip()
            txt = re.sub(r"^```(json)?|```$", "", txt, flags=re.MULTILINE).strip()
            return json.loads(txt)
        except json.JSONDecodeError:
            return {"methodology_stage":"parse_error","triples":[]}
        except Exception as e:
            if attempt == 3:
                return {"methodology_stage":"api_error","triples":[],"_err":str(e)[:120]}
            time.sleep(2 ** attempt)

def load_cache():
    done = {}
    if Path(LLM_CACHE).exists():
        for line in open(LLM_CACHE):
            try:
                d = json.loads(line); done[d["oa_id"]] = d
            except Exception:
                pass
    print(f"[llm] {len(done):,} cached", file=sys.stderr)
    return done

def llm_extract(df):
    cache = load_cache()
    if LLM_ON == "tick_only" and "llm_label" in df.columns:
        # work = df[df["llm_label"].astype(str).str.upper()=="TICK_RELEVANT"]
        work = df[df["llm_label"].notna()]
    elif LLM_ON == "sample":
        work = df.sample(min(LLM_SAMPLE_N, len(df)), random_state=42)
    else:
        work = df
    todo = work[~work["oa_id"].isin(cache.keys())]
    print(f"[llm] {len(todo):,} to process (of {len(work):,} in scope)", file=sys.stderr)

    def one(row):
        # Explicitly handle pd.NA values for title and abstract
        title_val = row.get("oa_title")
        abstract_val = row.get("oa_abstract")

        title = str(title_val) if pd.notna(title_val) else ""
        ab = str(abstract_val) if pd.notna(abstract_val) else ""

        if not ab:
            return {"oa_id":row["oa_id"],"openai":None,"deepseek":None,"skipped":"no_abstract"}
        return {"oa_id":row["oa_id"],
                "openai": call_llm("openai", title, ab),
                "deepseek": call_llm("deepseek", title, ab)}

    fout = open(LLM_CACHE, "a")
    rows = [r for _, r in todo.iterrows()]
    with ThreadPoolExecutor(max_workers=LLM_WORKERS) as ex:
        futs = {ex.submit(one, r): i for i, r in enumerate(rows)}
        for n, fut in enumerate(as_completed(futs), 1):
            d = fut.result()
            fout.write(json.dumps(d) + "\n"); fout.flush()
            cache[d["oa_id"]] = d
            if n % 200 == 0:
                print(f"[llm] {n:,}/{len(rows):,}", file=sys.stderr)
    fout.close()
    return cache


# =========================================================================== #
# 5. RECONCILE + WRITE                                                        #
# =========================================================================== #
def norm_triple(t):
    def c(x): return re.sub(r"\s+"," ",str(x or "").strip().lower())
    return (c(t.get("tick") or t.get("tick_species")),
            c(t.get("pathogen")), c(t.get("host")))

def reconcile(df, llm_cache):
    out = []
    for _, row in df.iterrows():
        oa = row["oa_id"]
        d_tick, d_path, d_host = dict_entities(row)
        d_stage, d_stageset = dict_methodology(row)
        d_trip = dict_triples(row)

        llm = llm_cache.get(oa, {})
        llm_stages, llm_trips = [], []
        for prov in ("openai","deepseek"):
            r = llm.get(prov) or {}
            if isinstance(r, dict):
                if r.get("methodology_stage"): llm_stages.append(r["methodology_stage"])
                for t in (r.get("triples") or []):
                    llm_trips.append(norm_triple(t))

        # stage consensus: prefer agreement among {dict, openai, deepseek}
        votes = Counter([d_stage] + [s for s in llm_stages if s not in
                        ("parse_error","api_error","")])
        stage_final = votes.most_common(1)[0][0] if votes else "other"
        stage_agree = (len(votes) and votes.most_common(1)[0][1] >= 2)

        # triple consensus: normalize, count sources
        tset = Counter()
        for t in d_trip:
            tset[norm_triple(t)] += 1
        for t in llm_trips:
            tset[t] += 1
        # keep triples seen by >=2 sources OR with all three axes filled
        strong = [t for t, n in tset.items()
                  if n >= 2 or all(t)]  # all(t) = tick&pathogen&host present

        out.append({
            "oa_id": oa,
            "meth_stage": stage_final,
            "meth_stage_agree": bool(stage_agree),
            "meth_stage_dict": d_stage,
            "meth_stage_set": ";".join(sorted(d_stageset)),
            "ent_ticks": ";".join(sorted(d_tick)),
            "ent_pathogens": ";".join(sorted(d_path)),
            "ent_hosts": ";".join(sorted(d_host)),
            "triples_strong": " || ".join(f"{a}|{b}|{c}" for a,b,c in strong),
            "n_triples_strong": len(strong),
        })
    res = pd.DataFrame(out)
    merged = df.merge(res, on="oa_id", how="left")
    merged.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}  ({len(merged):,} rows)", file=sys.stderr)
    return merged


# =========================================================================== #
# 6. VALIDATION SAMPLE (stage + triples, stratified by stage & region)        #
# =========================================================================== #
def make_validation(merged, path, n_per=40, seed=42):
    d = merged.copy()
    strat = d.get("study_bloc","" ).astype(str).str.split(";").str[0].fillna("none")
    d["_stratum"] = d["meth_stage"].astype(str) + "|" + strat
    parts = [g.sample(min(n_per,len(g)), random_state=seed) for _, g in d.groupby("_stratum")]
    s = pd.concat(parts)
    cols = ["oa_id","oa_year","oa_title","oa_abstract","meth_stage","meth_stage_agree",
            "ent_ticks","ent_pathogens","ent_hosts","triples_strong"]
    s = s[[c for c in cols if c in s.columns]].copy()
    s["gold_stage"] = ""; s["gold_triples"] = ""   # human fills
    s.to_csv(path, index=False)
    print(f"[validate] wrote {len(s):,} rows -> {path}", file=sys.stderr)


# =========================================================================== #
if __name__ == "__main__":
    df = pd.read_parquet(IN_PARQUET)
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    llm_cache = llm_extract(df) if RUN_LLM else {}
    merged = reconcile(df, llm_cache)
    make_validation(merged, OUT_PARQUET.replace(".parquet","_validation.csv"))


In [ ]:
df

In [ ]:
llm_cache

In [ ]:
merged